# COS 380: Lab 3
## TF-IDF and Document Similarity

Reuse your Lab 1 and Lab 2 modules. Complete the new reusable functions in `nlp_toolkit/weighting.py` and `nlp_toolkit/similarity.py`.

**Why TF-IDF?** Cosine similarity already cancels out document length differences, even on raw Bag-of-Words counts. What it does not fix is that every word is treated as equally important. TF-IDF fixes that second problem by weighting words based on how distinctive they are. This lab builds that improved representation and uses it to power a search engine.

## Task 1: Reconnect the Existing Pipeline

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

HERE = Path.cwd().resolve()
COURSE_ROOT = None
for folder in [HERE] + list(HERE.parents):
    if (folder / "nlp_toolkit").exists():
        COURSE_ROOT = folder
        break
if COURSE_ROOT is None:
    raise FileNotFoundError("Could not find the COS 380 folder containing nlp_toolkit.")
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

# Supports either course-folder name used this semester.
possible_paths = [
    COURSE_ROOT / "datasets" / "bbc_news" / "bbc_text_cls.xls",
    COURSE_ROOT / "data" / "bbc_news" / "bbc_text_cls.xls",
]
DATA_PATH = next((p for p in possible_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find bbc_text_cls.xls in datasets/bbc_news or data/bbc_news.")

bbc = pd.read_csv(DATA_PATH)
print("Dataset shape:", bbc.shape)

Dataset shape: (2225, 2)


In [2]:
from nlp_toolkit.preprocessing import (
    lowercase, remove_urls, remove_mentions, handle_hashtags,
    remove_punctuation, normalize_whitespace, tokenize,
    remove_stopwords, lemmatize_tokens,
)
from nlp_toolkit.vectorizer import build_vocabulary, create_word_to_index


def process_text(text):
    text = lowercase(text)
    text = remove_urls(text)
    text = remove_mentions(text)
    text = handle_hashtags(text)
    text = remove_punctuation(text)
    text = normalize_whitespace(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens, keep_negation=True)
    tokens = lemmatize_tokens(tokens)
    return tokens

In [3]:
WORKING_INDICES = [0, 1, 510, 511, 896, 897, 1313, 1314, 1824, 1825]
working = bbc.iloc[WORKING_INDICES].copy().reset_index(drop=True)
working["tokens"] = working["text"].apply(process_text)
tokenized_documents = working["tokens"].tolist()
vocabulary = build_vocabulary(tokenized_documents)
word_to_index = create_word_to_index(vocabulary)

print("Working labels:", working["labels"].tolist())
print("First 20 processed tokens:", tokenized_documents[0][:20])

Working labels: ['business', 'business', 'entertainment', 'entertainment', 'politics', 'politics', 'sport', 'sport', 'tech', 'tech']
First 20 processed tokens: ['ad', 'sale', 'boost', 'time', 'warner', 'profit', 'quarterly', 'profit', 'u', 'medium', 'giant', 'timewarner', 'jump', '76', '113bn', '£600m', 'three', 'month', 'december', '639m']


## Task 2: Compute Term Frequency
#### Implement `compute_tf(tokens)` in `weighting.py`, save the file, then run this check.

In [4]:
from nlp_toolkit.weighting import compute_tf

toy_tokens = ["cat", "cat", "sat"]
toy_tf = compute_tf(toy_tokens)
print(toy_tf)
print("TF(cat):", toy_tf["cat"])
print("TF(sat):", toy_tf["sat"])

{'cat': 0.6666666666666666, 'sat': 0.3333333333333333}
TF(cat): 0.6666666666666666
TF(sat): 0.3333333333333333


## Task 3: Compute IDF
#### Implement `compute_idf(tokenized_documents)` using `ln(N / DF)`.

In [5]:
from nlp_toolkit.weighting import compute_idf

toy_docs = [["cat", "sat"], ["cat", "slept"]]
toy_idf = compute_idf(toy_docs)
print(toy_idf)
print("cat lower than sat:", toy_idf["cat"] < toy_idf["sat"])
print("cat lower than slept:", toy_idf["cat"] < toy_idf["slept"])

{'cat': 0.0, 'sat': 0.6931471805599453, 'slept': 0.6931471805599453}
cat lower than sat: True
cat lower than slept: True


## Task 4: Build TF-IDF Vectors
#### Implement `tfidf_document()` and `tfidf_corpus()` in `weighting.py`.

In [6]:
from nlp_toolkit.weighting import tfidf_document, tfidf_corpus

idf = compute_idf(tokenized_documents)
X_tfidf = tfidf_corpus(tokenized_documents, word_to_index, idf)
print("TF-IDF matrix shape:", X_tfidf.shape)

TF-IDF matrix shape: (10, 1085)


## Task 5: Compute Cosine Similarity
#### Implement `cosine_similarity()` in `similarity.py`.

Cosine similarity measures the angle between two vectors, not the distance between them, so it already cancels out document length differences on its own. That is why you are not asked to normalize your vectors separately before comparing them.

In [7]:
from nlp_toolkit.similarity import cosine_similarity

print("Identical:", cosine_similarity([1, 0], [1, 0]))
print("Orthogonal:", cosine_similarity([1, 0], [0, 1]))
print("Zero vector:", cosine_similarity([0, 0], [1, 0]))

Identical: 1.0
Orthogonal: 0
Zero vector: 0


## Task 6 :Build the BBC Article Search Engine
#### Implement `find_similar_documents()` and use one article as the query.

In [8]:
from nlp_toolkit.similarity import find_similar_documents

query_index = 0
results = find_similar_documents(query_index, X_tfidf, top_k=3)

print("QUERY")
print("Label:", working.loc[query_index, "labels"])
print("Excerpt:", working.loc[query_index, "text"][:300], "...")
print()

print("TOP 3 RESULTS")
for rank, (doc_index, score) in enumerate(results, start=1):
    print(f"{rank}. Document {doc_index}")
    print("   Label:", working.loc[doc_index, "labels"])
    print(f"   Similarity: {score:.4f}")
    print("   Excerpt:", working.loc[doc_index, "text"][:250], "...")
    print()

QUERY
Label: business
Excerpt: Ad sales boost Time Warner profit

Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.

The firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and highe ...

TOP 3 RESULTS
1. Document 1
   Label: business
   Similarity: 0.0498
   Excerpt: Dollar gains on Greenspan speech

The dollar has hit its highest level against the euro in almost three months after the Federal Reserve head said the US trade deficit is set to stabilise.

And Alan Greenspan highlighted the US government's willingne ...

2. Document 4
   Label: politics
   Similarity: 0.0256
   Excerpt: Labour plans maternity pay rise

Maternity pay for new mothers is to rise by £1,400 as part of new proposals announced by the Trade and Industry Secretary Patricia Hewitt.

It would mean paid leave would be increased to nine months by 2007, Ms Hewitt ...

3. 

**Brief response:** Are the retrieved articles meaningfully related to the query? Explain using the article text, not only the category labels.

**Your response:**

The retrieved articles are somewhat related to the query. The Query is an article about quarterly profits increasing at Time Warner. The most related article is about the value of the dollar increasing. The next article, in a similar vein, is about maternity leave pay rising. This makes since because they all both about money and things going up! In a corpus of only 10 documents, it is not surprising that these are not very similar other than those qualities. 

## Task 7 :Compare CountVectorizer with TF-IDF



In [9]:
from nlp_toolkit.vectorizer import vectorize_corpus
from nlp_toolkit.similarity import find_similar_documents

X_bow = vectorize_corpus(tokenized_documents, word_to_index)
bow_results = find_similar_documents(query_index, X_bow, top_k=3)

print("BAG-OF-WORDS TOP 3")
for rank, (doc_index, score) in enumerate(bow_results, start=1):
    print(f"{rank}. Document {doc_index} | {working.loc[doc_index, 'labels']} | {score:.4f}")

print()
print("TF-IDF TOP 3 (from Task 6 above)")
for rank, (doc_index, score) in enumerate(results, start=1):
    print(f"{rank}. Document {doc_index} | {working.loc[doc_index, 'labels']} | {score:.4f}")

BAG-OF-WORDS TOP 3
1. Document 1 | business | 0.1607
2. Document 4 | politics | 0.1281
3. Document 5 | politics | 0.1009

TF-IDF TOP 3 (from Task 6 above)
1. Document 1 | business | 0.0498
2. Document 4 | politics | 0.0256
3. Document 9 | tech | 0.0121


In [10]:
indices_to_view = {query_index}

for doc_index, score in bow_results:
    indices_to_view.add(doc_index)

for doc_index, score in results:
    indices_to_view.add(doc_index)

# Display the relevant articles
for i in sorted(indices_to_view):
    print("=" * 80)

    if i == query_index:
        print(f"QUERY DOCUMENT — Index {i}")
    else:
        print(f"DOCUMENT INDEX: {i}")

    print(f"CATEGORY: {working.loc[i, 'labels']}")
    print()
    print(working.loc[i, "text"])
    print()

QUERY DOCUMENT — Index 0
CATEGORY: business

Ad sales boost Time Warner profit

Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.

The firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and higher advert sales. TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn. Its profits were buoyed by one-off gains which offset a profit dip at Warner Bros, and less users for AOL.

Time Warner said on Friday that it now owns 8% of search-engine Google. But its own internet business, AOL, had has mixed fortunes. It lost 464,000 subscribers in the fourth quarter profits were lower than in the preceding three quarters. However, the company said AOL's underlying profit before exceptional items rose 8% on the back of stronger internet advertising revenues. It hopes to increase subscribers by offering the online service free to TimeWarner i

**TO DO** Compare the Bag-of-Words and TF-IDF search results. Which rankings changed? Why might TF-IDF’s way of assigning importance to words cause the rankings to change?

Although both methods ranked the same articles 1 and 2, they differed on the 3rd article. We see changes in the rankings because TF-IDF normalizes for words that are very common. It assigns higher weight to unique words, while Bag of Words gets lets words like "the" and "say" drag up a similarity ranking, even when they aren't important.

## Submit
- `labs/lab03/Lab03.ipynb`
- `nlp_toolkit/weighting.py`
- `nlp_toolkit/similarity.py`

Do not modify or submit `course_tools/`.